In [1]:
import pandas as pd 
import re
import os
import yaml
from pathlib import Path
import ast

Create some Directory to be used

In [2]:
cleaned_dir = Path("PROCESSED")

cleaned_dir.mkdir(parents = True, exist_ok = True)

Load the Dataset Using Pandas

In [3]:
with open("dataconf.yml", "r") as f:
    config = yaml.safe_load(f)
file_path = os.path.join(config["data_path"], "dum_data.xlsx")

#print(f"File path: {file_path}")

In [4]:
file_path = Path("basedata/dumm_data.xlsx").resolve()
df = pd.read_excel(file_path, skiprows = 0)

In [5]:
df.head(25)

,Timestamp,Datetime,Procada Current,Procada Voltage,Laser Power,Wire Current,Wire Speed,Wire Voltage,Unnamed: 8,X,Y,Z,<- Robot pos
0,1.704451e+09,2024-01-05 10:34:52.162,"{'SetVal': 0.0, 'v': 0.2871}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",NaN,-482.449829,-14.914701,151.575500,NaN
1,1.704451e+09,2024-01-05 10:34:52.195,"{'SetVal': 0.0, 'v': 0.2871}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",NaN,-482.263367,-19.762871,151.791931,NaN
2,1.704451e+09,2024-01-05 10:34:52.229,"{'SetVal': 0.0, 'v': 0.2871}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",NaN,-483.006927,-16.474230,151.901566,NaN
3,1.704451e+09,2024-01-05 10:34:52.262,"{'SetVal': 0.0, 'v': 0.2871}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",NaN,-480.737061,-17.901360,135.418030,NaN
4,1.704451e+09,2024-01-05 10:34:52.295,"{'SetVal': 0.0, 'v': 0.2893}","{'SetVal': 0.0, 'v': 0.0002}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",NaN,-480.826111,-16.183176,135.999893,NaN
5,1.704451e+09,2024-01-05 10:34:52.329,"{'SetVal': 0.0, 'v': 0.2893}","{'SetVal': 0.0, 'v': 0.0002}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",NaN,-483.828003,-18.554438,152.221420,NaN
6,1.704451e+09,2024-01-05 10:34:52.362,"{'SetVal': 0.0, 'v': 0.2893}","{'SetVal': 0.0, 'v': 0.0002}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",NaN,-480.990173,-16.998444,136.692535,NaN
7,1.704451e+09,2024-01-05 10:34:52.395,"{'SetVal': 0.0, 'v': 0.2802}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",NaN,-484.385254,-14.036122,151.920593,NaN
8,1.704451e+09,2024-01-05 10:34:52.429,"{'SetVal': 0.0, 'v': 0.2802}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",NaN,-485.242981,-16.541672,151.891312,NaN
9,1.704451e+09,2024-01-05 10:34:52.462,"{'SetVal': 0.0, 'v': 0.2802}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",NaN,-483.984833,-14.099407,151.901077,NaN


`Cleaning Process`

In [6]:
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace("<- Robot pos", "")     # Remove the "<- Robot pos"


df = df.iloc[:, :-1]

In [7]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')

In [8]:
df = df.drop(columns=[col for col in df.columns if 'Unnamed' in col or 'empty' in col.lower()])

In [9]:
df.head(10)

,Timestamp,Datetime,Procada Current,Procada Voltage,Laser Power,Wire Current,Wire Speed,Wire Voltage,X,Y,Z
0,2024-01-05 10:34:52.161999941,2024-01-05 10:34:52.162,"{'SetVal': 0.0, 'v': 0.2871}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",-482.449829,-14.914701,151.575500
1,2024-01-05 10:34:52.195333004,2024-01-05 10:34:52.195,"{'SetVal': 0.0, 'v': 0.2871}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",-482.263367,-19.762871,151.791931
2,2024-01-05 10:34:52.228667021,2024-01-05 10:34:52.229,"{'SetVal': 0.0, 'v': 0.2871}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",-483.006927,-16.474230,151.901566
3,2024-01-05 10:34:52.262000084,2024-01-05 10:34:52.262,"{'SetVal': 0.0, 'v': 0.2871}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",-480.737061,-17.901360,135.418030
4,2024-01-05 10:34:52.295332909,2024-01-05 10:34:52.295,"{'SetVal': 0.0, 'v': 0.2893}","{'SetVal': 0.0, 'v': 0.0002}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",-480.826111,-16.183176,135.999893
5,2024-01-05 10:34:52.328666925,2024-01-05 10:34:52.329,"{'SetVal': 0.0, 'v': 0.2893}","{'SetVal': 0.0, 'v': 0.0002}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",-483.828003,-18.554438,152.221420
6,2024-01-05 10:34:52.361999989,2024-01-05 10:34:52.362,"{'SetVal': 0.0, 'v': 0.2893}","{'SetVal': 0.0, 'v': 0.0002}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",-480.990173,-16.998444,136.692535
7,2024-01-05 10:34:52.395333052,2024-01-05 10:34:52.395,"{'SetVal': 0.0, 'v': 0.2802}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",-484.385254,-14.036122,151.920593
8,2024-01-05 10:34:52.428667068,2024-01-05 10:34:52.429,"{'SetVal': 0.0, 'v': 0.2802}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",-485.242981,-16.541672,151.891312
9,2024-01-05 10:34:52.461999893,2024-01-05 10:34:52.462,"{'SetVal': 0.0, 'v': 0.2802}","{'SetVal': 0.0, 'v': 0.0001}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 0.0, 'v': 0.0}","{'SetVal': 5.0, 'v': 5.0}","{'SetVal': 0.0, 'v': 0.0}",-483.984833,-14.099407,151.901077


Extract Numeric values from the SetVal strings.

In [10]:
def extract_numvalue(value):
    """_summary_
    Convert the string representation of dictionary to actual dictionary
    First, replace single quotes with double quotes for proper JSON format

    Returns:
        float: Value of each measured quatities in the data (the 'v' value from the dictionary)
    """
    
    if isinstance(value, str):
        if "SetVal" in value:
            try:
                value_dict = ast.literal_eval(value)

                return float(value_dict['v'])
            except (ValueError, SyntaxError, KeyError):
                # Fallback to regex if the above method fails
                match = re.search(r"'v': '?([-\d.]+)'?", value)
                if match:
                    return float(match.group(1))
        try:
            return float(value)
        except ValueError:
            return value
    return value

Apply the Cleaning to all columns except DateTime

In [11]:
numeric_columns = df.columns.difference(["Datetime", "Timestamp"])
for col in numeric_columns:
    df[col] = df[col].apply(extract_numvalue)

Convert the Datatime to proper datatime format.

In [12]:
df['Datetime'] = pd.to_datetime(df["Datetime"])

In [13]:
df = df.dropna(how = "all")

#Reset the index.
df = df.reset_index(drop = True)

Check for missing values.

In [14]:
print(f"Missing values: \n {df.isnull().sum()}")

Missing values: 
 Timestamp          0
Datetime           0
Procada Current    0
Procada Voltage    0
Laser Power        0
Wire Current       0
Wire Speed         0
Wire Voltage       0
X                  0
Y                  0
Z                  0
dtype: int64


In [15]:
df.head(50)

,Timestamp,Datetime,Procada Current,Procada Voltage,Laser Power,Wire Current,Wire Speed,Wire Voltage,X,Y,Z
0,2024-01-05 10:34:52.161999941,2024-01-05 10:34:52.162,0.2871,0.0001,0.0,0.0,5.00,0.00,-482.449829,-14.914701,151.575500
1,2024-01-05 10:34:52.195333004,2024-01-05 10:34:52.195,0.2871,0.0001,0.0,0.0,5.00,0.00,-482.263367,-19.762871,151.791931
2,2024-01-05 10:34:52.228667021,2024-01-05 10:34:52.229,0.2871,0.0001,0.0,0.0,5.00,0.00,-483.006927,-16.474230,151.901566
3,2024-01-05 10:34:52.262000084,2024-01-05 10:34:52.262,0.2871,0.0001,0.0,0.0,5.00,0.00,-480.737061,-17.901360,135.418030
4,2024-01-05 10:34:52.295332909,2024-01-05 10:34:52.295,0.2893,0.0002,0.0,0.0,5.00,0.00,-480.826111,-16.183176,135.999893
5,2024-01-05 10:34:52.328666925,2024-01-05 10:34:52.329,0.2893,0.0002,0.0,0.0,5.00,0.00,-483.828003,-18.554438,152.221420
6,2024-01-05 10:34:52.361999989,2024-01-05 10:34:52.362,0.2893,0.0002,0.0,0.0,5.00,0.00,-480.990173,-16.998444,136.692535
7,2024-01-05 10:34:52.395333052,2024-01-05 10:34:52.395,0.2802,0.0001,0.0,0.0,5.00,0.00,-484.385254,-14.036122,151.920593
8,2024-01-05 10:34:52.428667068,2024-01-05 10:34:52.429,0.2802,0.0001,0.0,0.0,5.00,0.00,-485.242981,-16.541672,151.891312
9,2024-01-05 10:34:52.461999893,2024-01-05 10:34:52.462,0.2802,0.0001,0.0,0.0,5.00,0.00,-483.984833,-14.099407,151.901077


In [16]:
df.tail(50)

,Timestamp,Datetime,Procada Current,Procada Voltage,Laser Power,Wire Current,Wire Speed,Wire Voltage,X,Y,Z
950,2024-01-05 10:35:23.828666925,2024-01-05 10:35:23.829,0.2752,0.0043,0.0,0.0,0.0,0.0,-419.081146,246.394669,152.975449
951,2024-01-05 10:35:23.861999989,2024-01-05 10:35:23.862,0.2752,0.0043,0.0,0.0,0.0,0.0,-418.041016,248.115417,153.013336
952,2024-01-05 10:35:23.895333052,2024-01-05 10:35:23.895,0.2752,0.0043,0.0,0.0,0.0,0.0,-307.223175,376.942352,154.410522
953,2024-01-05 10:35:23.928667068,2024-01-05 10:35:23.929,0.2752,0.0043,0.0,0.0,0.0,0.0,-318.031097,385.642639,128.519577
954,2024-01-05 10:35:23.961999893,2024-01-05 10:35:23.962,0.2752,0.0043,0.0,0.0,0.0,0.0,-304.707245,377.329926,154.211166
955,2024-01-05 10:35:23.995332956,2024-01-05 10:35:23.995,0.2752,0.0043,0.0,0.0,0.0,0.0,-300.202789,374.981201,150.829865
956,2024-01-05 10:35:24.028666019,2024-01-05 10:35:24.029,0.2752,0.0043,0.0,0.0,0.0,0.0,-319.495239,384.656525,125.975632
957,2024-01-05 10:35:24.062000036,2024-01-05 10:35:24.062,0.2752,0.0043,0.0,0.0,0.0,0.0,-305.560394,379.242737,154.452774
958,2024-01-05 10:35:24.095333099,2024-01-05 10:35:24.095,0.2752,0.0043,0.0,0.0,0.0,0.0,-303.844147,379.458008,154.369537
959,2024-01-05 10:35:24.128667116,2024-01-05 10:35:24.129,0.2752,0.0043,0.0,0.0,0.0,0.0,-322.072357,384.867493,124.109779


In [17]:
print("Basic statistics:\n")

df.describe()

Basic statistics:



,Timestamp,Datetime,Procada Current,Procada Voltage,Laser Power,Wire Current,Wire Speed,Wire Voltage,X,Y,Z
count,1000,1000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,2024-01-05 10:35:08.812000,2024-01-05 10:35:08.812000,28.965345,1.765583,3829.398000,0.044600,2.461910,0.017840,-430.759930,180.087668,151.534939
min,2024-01-05 10:34:52.161999941,2024-01-05 10:34:52.162000,0.134100,0.000100,0.000000,0.000000,0.000000,0.000000,-491.213104,-21.173449,96.233009
25%,2024-01-05 10:35:00.486999552,2024-01-05 10:35:00.486749952,0.275200,0.004300,0.000000,0.000000,0.000000,0.000000,-476.894089,81.326271,151.504318
50%,2024-01-05 10:35:08.811999744,2024-01-05 10:35:08.812000,0.275200,0.004300,0.000000,0.000000,0.000000,0.000000,-453.475479,170.617676,152.867928
75%,2024-01-05 10:35:17.137000448,2024-01-05 10:35:17.137250048,71.136600,3.863300,9020.000000,0.100000,5.240000,0.040000,-398.811897,278.328461,153.751873
max,2024-01-05 10:35:25.461999893,2024-01-05 10:35:25.462000,76.389700,5.039200,9020.000000,0.100000,5.240000,0.040000,-286.253357,393.276672,157.679810
std,NaN,NaN,35.023862,1.989290,4420.988124,0.049732,2.600706,0.019893,55.500686,118.575226,6.431179


In [18]:
file_path = cleaned_dir/"cleaned_proc_data_v1.csv"
df.to_csv(file_path, index = False)

In [19]:
file_path = cleaned_dir/"cleaned_proc_data_v1.xlsx"

df.to_excel(file_path, index = False)